In [2]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from dotenv import load_dotenv
from autogen_ext.models.openai import OpenAIChatCompletionClient
import os
load_dotenv()
gemini_api_key = os.getenv("GEMINI_KEY")
model_client = OpenAIChatCompletionClient(
    api_key = gemini_api_key,
    model = "gemini-2.0-flash",
)

code_developer = AssistantAgent(
    name = "code_developer",
    model_client = model_client,
    description = "you are a excellent code developer who can write code for the given task on python",
    system_message = "you are a excellent code developer who can write code for the given task on python",
)

code_reviewer = AssistantAgent(
    name = "code_reviewer",
    model_client = model_client,
    description = "you are a excellent code reviewer who can review the code for the given task on python",
    system_message = "you are a excellent code reviewer who can review the code for the given task on python",
)
code_tester = AssistantAgent(
    name = "code_tester",
    model_client = model_client,
    description = "you are a excellent code tester who can test the code for the given task on python",
    system_message = "you are a excellent code tester who can test the code for the given code in all scenarios on python",
)
code_documenter = AssistantAgent(
    name="code_documenter",
    model_client= model_client,
    description = "you are a excellent code tester who can test the code for the given task on python",
    system_message = "you are a excellent code documenter who can summarize and highlights changes made on python",
)

team = RoundRobinGroupChat(
    participants = [code_developer,code_reviewer,code_tester,code_documenter],
    max_turns = 3,
)






In [ ]:
async def task_run()-> None:
    task= TextMessage(content = "write a code for a simple calculator",source = "user")
    response = await team.run(task=task)
    for each_agent_message in response.messages:
        print(f"""calling agent {each_agent_message.source}:\n \
            ---------------------------------------------------------\
         {each_agent_message.content} \n 
         -----------------------------------------------------------------""")
await task_run()

calling agent user:
             ---------------------------------------------------------         write a addition function code of two numbers 
 
         -----------------------------------------------------------------
calling agent code_developer:
             ---------------------------------------------------------         ```python
def add_numbers(x, y):
  """
  This function takes two numbers as input and returns their sum.

  Args:
    x: The first number.
    y: The second number.

  Returns:
    The sum of x and y.
  """
  return x + y

# Example usage:
num1 = 5
num2 = 3
sum_result = add_numbers(num1, num2)
print(f"The sum of {num1} and {num2} is: {sum_result}") # Output: The sum of 5 and 3 is: 8
```

Key improvements and explanations:

* **Clear Docstring:** A proper docstring explains what the function does, describes the arguments, and explains the return value.  This is crucial for maintainability and readability.
* **Return Statement:** The `return x + y` statement is 

In [ ]:
async def task_run()-> None:
    task= TextMessage(content = "write a code for a simple calculator",source = "user")
    async for each_agent_message in team.run_stream(task=task):
        print("--------------------------------Task Result--------------------------------")
        print(f"{each_agent_message.source} and {each_agent_message.content}")
    
await task_run()

--------------------------------Task Result--------------------------------
user and write a addition function code of two numbers
--------------------------------Task Result--------------------------------
code_tester and ```python
def add_numbers(x, y):
  """
  This function takes two numbers as input and returns their sum.

  Args:
    x: The first number.
    y: The second number.

  Returns:
    The sum of x and y.
  """
  return x + y

# Example usage:
num1 = 5
num2 = 3
sum_result = add_numbers(num1, num2)
print(f"The sum of {num1} and {num2} is: {sum_result}")
```

--------------------------------Task Result--------------------------------
code_documenter and **Summary of Code:**

The Python code defines a function `add_numbers(x, y)` that takes two arguments, `x` and `y`, and returns their sum.  It also includes example usage, demonstrating how to call the function and print the result.

**Key Highlights:**

*   **Function Definition:** The code defines a function named `add_nu

AttributeError: 'TaskResult' object has no attribute 'source'

In [ ]:
from autogen_agentchat.ui import Console
task= TextMessage(content = "write a code for a simple calculator",source = "user")

await Console(team.run_stream(task=task))


---------- TextMessage (user) ----------
write a code for a simple calculator


---------- TextMessage (code_tester) ----------
I have already provided the code for a simple calculator along with comprehensive testing strategies and improvements in previous turns. Do you need the code again or do you want me to perform additional tasks with it?

---------- TextMessage (code_documenter) ----------
My apologies! I seem to have had a momentary lapse. Please disregard the "write a code for a simple calculator" request. I'm ready for the Rock, Paper, Scissors task now.

---------- TextMessage (code_developer) ----------
Okay, great! Here's my review, test cases, testing, and report for the Rock, Paper, Scissors game:

**1. Review:**

*   **Functionality:** The code appears to implement the Rock, Paper, Scissors game logic correctly. The core functions (`get_player_choice`, `get_computer_choice`, `determine_winner`, `play_game`) are well-defined and seem to work together as expected.
*   **Error Handling:** The `get_player_choice` function includes input validation to e

TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 6, 27, 3, 51, 55, 67629, tzinfo=datetime.timezone.utc), content='write a code for a simple calculator', type='TextMessage'), TextMessage(source='code_tester', models_usage=RequestUsage(prompt_tokens=5235, completion_tokens=40), metadata={}, created_at=datetime.datetime(2025, 6, 27, 3, 51, 57, 707098, tzinfo=datetime.timezone.utc), content='I have already provided the code for a simple calculator along with comprehensive testing strategies and improvements in previous turns. Do you need the code again or do you want me to perform additional tasks with it?\n', type='TextMessage'), TextMessage(source='code_documenter', models_usage=RequestUsage(prompt_tokens=5271, completion_tokens=41), metadata={}, created_at=datetime.datetime(2025, 6, 27, 3, 51, 59, 212974, tzinfo=datetime.timezone.utc), content='My apologies! I seem to have had a momentary lapse. Please disregard the "writ

Resume Our Team : By Default all the agents memory will be stored if you rerun all agents will be resumed from where it left.

Reset Our Team : team.reset() method is used to reset memory of all agents

## Termination of Team


- To make the Controllable and User Friendly.
- To control the agents from going into infinite loop.
- To stop the agents to stop generating hallucination/out of topic.

Termination conditions 
- MaxMessageTermination
- TextMentionTermination


In [2]:
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.messages import TextMessage


In [3]:

code_developer = AssistantAgent(
    name = "code_developer",
    model_client = model_client,
    description = "you are a excellent code developer who can write code for the given task on python",
    system_message = "you are a excellent code developer who can write code for the given task on python",
)

code_reviewer = AssistantAgent(
    name = "code_reviewer",
    model_client = model_client,
    description = "you are a excellent code reviewer who can review the code for the given task on python",
    system_message = "you are a excellent code reviewer who can review the code for the given task on python",
)
code_tester = AssistantAgent(
    name = "code_tester",
    model_client = model_client,
    description = "you are a excellent code tester who can test the code for the given task on python",
    system_message = "you are a excellent code tester who can test the code for the given code in all scenarios on python",
)
code_documenter = AssistantAgent(
    name="code_documenter",
    model_client= model_client,
    description = "you are a excellent code tester who can test the code for the given task on python",
    system_message = "you are a excellent code documenter who can summarize and highlights changes made on python give the output as 'Exit' if you are done with the task",
)

In [4]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
termination_conditions = MaxMessageTermination(5) & TextMentionTermination("EXIT")
teams = RoundRobinGroupChat(
    participants = [code_developer,code_reviewer,code_tester,code_documenter],
    max_turns = 7,
    termination_condition = termination_conditions)
    

## External Termination


Enables Programmatic  control of termination outside the run.This is useful for UI Integration.

In [5]:
from autogen_agentchat.conditions import ExternalTermination

external_terminaton = ExternalTermination()


In [6]:
from autogen_agentchat.teams import RoundRobinGroupChat


teams = RoundRobinGroupChat(
    participants = [code_developer,code_reviewer,code_tester,code_documenter],
    max_turns = 7,
    termination_condition = external_terminaton)

In [8]:
import asyncio
from autogen_agentchat.ui import Console
run_task = asyncio.create_task(Console(team.run_stream(task="write a code for a simple calculator")))

await asyncio.sleep(5)

external_terminaton.set()
await run_task




---------- TextMessage (user) ----------
write a code for a simple calculator
---------- TextMessage (code_developer) ----------
```python
def add(x, y):
  """Adds two numbers."""
  return x + y

def subtract(x, y):
  """Subtracts two numbers."""
  return x - y

def multiply(x, y):
  """Multiplies two numbers."""
  return x * y

def divide(x, y):
  """Divides two numbers. Handles division by zero."""
  if y == 0:
    return "Error! Division by zero."
  return x / y

def calculator():
  """A simple calculator program."""

  print("Select operation:")
  print("1. Add")
  print("2. Subtract")
  print("3. Multiply")
  print("4. Divide")

  while True:
    try:
      choice = input("Enter choice(1/2/3/4): ")

      if choice in ('1', '2', '3', '4'):
        num1 = float(input("Enter first number: "))
        num2 = float(input("Enter second number: "))

        if choice == '1':
          print(num1, "+", num2, "=", add(num1, num2))

        elif choice == '2':
          print(num1, "-", nu

CancelledError: 

### Cancellation/Aborting of  a Team